In [ ]:
# ====================================================================
#  SMART BUS TICKET BOOKING & RESERVATION SYSTEM
#  A text-based (console) Python 3 application.
#  Built-in Python only. All records are kept in memory, so they are
#  lost when the program ends, nothing is saved in any way or in data base.
# ====================================================================


# --------------------------------------------------------------------
# SECTION 1: DATA
# --------------------------------------------------------------------

SUPPORTED_CITIES = ("Karachi", "Hyderabad", "peshawar", "islamabad", "Quetta")

# List of dictionaries: one dictionary per bus route.
# Fares are in rupees(integers).
routes = [
    {
        "route_id": "R101",
        "departure_city": "Karachi",
        "destination_city": "quetta",
        "departure_time": "07:00 AM",
        "ticket_price": 1200,
        "total_seats": 40,
        "available_seats": 40
    },
    {
        "route_id": "R102",
        "departure_city": "Karachi",
        "destination_city": "Sukkur",
        "departure_time": "09:30 AM",
        "ticket_price": 2000,
        "total_seats": 36,
        "available_seats": 36
    },
    {
        "route_id": "R103",
        "departure_city": "Hyderabad",
        "destination_city": "Mirpur Khas",
        "departure_time": "11:00 AM",
        "ticket_price": 800,
        "total_seats": 32,
        "available_seats": 32
    },
    {
        "route_id": "R104",
        "departure_city": "Karachi",
        "destination_city": "Lahore",
        "departure_time": "07:00 PM",
        "ticket_price": 5000,
        "total_seats": 45,
        "available_seats": 45
    },
    {
        "route_id": "R105",
        "departure_city": "Sukkur",
        "destination_city": "Hyderabad",
        "departure_time": "02:00 PM",
        "ticket_price": 1500,
        "total_seats": 30,
        "available_seats": 30
    }
]

# List of dictionaries: one dictionary per booking.
# Cancelled bookings stay in this list (history is never deleted).
bookings = []

# Set: every booking ID we have issued. A set cannot hold duplicates and
# checking "is this ID already used?" is quick, so it protects unique IDs.
issued_booking_ids = set()

# Note: the functions below change these three collections (append, add,
# or edit a value inside them). They never assign a brand-new value to
# the names, so the "global" keyword is not needed.


# --------------------------------------------------------------------
# SECTION 2: SMALL HELPER FUNCTIONS
# --------------------------------------------------------------------

def find_route(route_id):
    # Look through all routes for one whose ID matches.
    for route in routes:
        if route["route_id"] == route_id:
            found_route = route
            break  # stop searching as soon as the route is found
    else:
        # for-else: this "else" belongs to the "for". It runs only when the
        # loop finished WITHOUT hitting "break", meaning no route matched.
        found_route = None
    return found_route


def generate_booking_id():
    # Start at B1001 and move forward until we find an ID not issued yet.
    number = 1001
    new_id = "B" + str(number)   # str() turns the number into text so it can be joined
    while new_id in issued_booking_ids:   # check the set before using an ID
        number = number + 1
        new_id = "B" + str(number)
    issued_booking_ids.add(new_id)   # remember it so it is never issued again
    return new_id


def is_positive_whole_number(text):
    # Returns True only when the text is a whole number greater than zero.
    # isdecimal() is True only for text made of digits, so "abc", "2.5",
    # "-3" and "" are rejected here BEFORE int() is ever called.
    if not text.isdecimal():
        return False
    if int(text) <= 0:
        return False
    return True


def describe_journey(route):
    # Example result: "Karachi -> Hyderabad"
    return route["departure_city"] + " -> " + route["destination_city"]


def print_route_line(route):
    # One-line summary of a route, used by the search feature.
    if route["available_seats"] == 0:
        print(route["route_id"], "|", describe_journey(route), "| Rs.",
              route["ticket_price"], "| 0 seats available (Fully Booked)")
    else:
        print(route["route_id"], "|", describe_journey(route), "| Rs.",
              route["ticket_price"], "|", route["available_seats"], "seats available")


# --------------------------------------------------------------------
# SECTION 3: MENU
# --------------------------------------------------------------------

def show_menu():
    print()
    print("SMART BUS RESERVATION SYSTEM")
    print("=" * 32)
    print("1. View Available Routes")
    print("2. Search for a Route")
    print("3. Book a Ticket")
    print("4. View My Bookings")
    print("5. Cancel a Booking")
    print("6. View Seat Availability")
    print("7. Generate Booking Report")
    print("8. Exit")
    print("=" * 32)


# --------------------------------------------------------------------
# SECTION 4: FEATURES (one function per menu action)
# --------------------------------------------------------------------

# ---- Menu option 1: View Available Routes --------------------------

def show_routes(only_available):
    # only_available is a Boolean: True hides fully booked routes.
    print()
    print("AVAILABLE BUS ROUTES")
    print("No. | Route | Journey | Depart | Fare | Seats")

    shown_count = 0
    # enumerate() gives each route a number (starting at 1) along with the route.
    for number, route in enumerate(routes, start=1):
        if only_available and route["available_seats"] == 0:
            continue  # skip this fully booked route and go to the next one

        shown_count = shown_count + 1
        seats_text = str(route["available_seats"]) + "/" + str(route["total_seats"])
        if route["available_seats"] == 0:
            seats_text = seats_text + " Fully Booked"
        print(number, "|", route["route_id"], "|", describe_journey(route), "|",
              route["departure_time"], "| Rs." + str(route["ticket_price"]), "|", seats_text)

    if shown_count == 0:
        print("No routes with seats available.")


def show_schedule():
    print()
    print("ROUTE SCHEDULE BY DEPARTURE CITY")
    # Outer loop: visit every supported city.
    for city in SUPPORTED_CITIES:
        print(city)
        has_departure = False  # Boolean flag: did we print any route for this city?
        # Inner loop: check every route against the current city.
        for route in routes:
            if route["departure_city"] == city:
                print("    " + route["route_id"], "->", route["destination_city"],
                      "|", route["departure_time"])
                has_departure = True
        if not has_departure:
            print("    No scheduled departures.")


def show_routes_menu():
    # Option 1 opens this small sub-menu (the PRD has no separate menu slot
    # for the schedule, so it lives here).
    print()
    print("VIEW ROUTES")
    print("1. All routes")
    print("2. Only routes with seats available")
    print("3. Route schedule by departure city")
    choice = input("Enter your choice: ").strip()
    if choice == "1":
        show_routes(False)
    elif choice == "2":
        show_routes(True)
    elif choice == "3":
        show_schedule()
    else:
        print("Invalid choice. Please enter 1, 2 or 3.")


# ---- Menu option 2: Search for a Route -----------------------------

def search_routes():
    term = input("Search by route ID or city: ").strip()
    if term == "":
        print("Please type a route ID or a city name.")
        return

    route_id_text = term.upper()   # route IDs are stored in capitals, e.g. R101
    city_text = term.lower()       # lower() makes the city search case-insensitive
    print()

    # Step 1: try to match a route ID. A route ID is unique, so we stop at the first match.
    for route in routes:
        if route["route_id"] == route_id_text:
            print_route_line(route)
            break
    else:
        # for-else: we only reach this "else" when the loop above ended WITHOUT
        # "break", i.e. no route ID matched. So we treat the text as a city name.
        # A city can match several routes, so there is no "break" here; we
        # count the matches instead and print the no-match message if the count is 0.
        match_count = 0
        for route in routes:
            if (route["departure_city"].lower() == city_text
                    or route["destination_city"].lower() == city_text):
                print_route_line(route)
                match_count = match_count + 1
        if match_count == 0:
            print("No matching routes found.")


# ---- Menu option 3: Book a Ticket ----------------------------------

def book_ticket():
    print()
    print("BOOK A TICKET")

    # Validate each field BEFORE anything is changed. If a check fails we
    # print a message and return, so no booking is made and no seats change.
    name = input("Enter passenger name: ").strip()
    if name == "":
        print("Booking failed: passenger name cannot be empty.")
        return

    age_text = input("Enter passenger age: ").strip()
    if not is_positive_whole_number(age_text):
        print("Booking failed: age must be a positive whole number.")
        return
    age = int(age_text)

    route_id = input("Enter route ID (for example R101): ").strip().upper()
    route = find_route(route_id)

    # Nested conditions: route exists -> seats entered correctly -> enough seats.
    if route is None:
        print("Booking failed: route", route_id, "does not exist.")
    elif route["available_seats"] == 0:
        print("Booking failed: route", route_id, "is fully booked.")
    else:
        seats_text = input("Enter number of seats: ").strip()
        if not is_positive_whole_number(seats_text):
            print("Booking failed: seats must be a positive whole number.")
        else:
            seats = int(seats_text)
            if seats > route["available_seats"]:
                print("Booking failed: only", route["available_seats"],
                      "seat(s) available on route", route_id + ".")
            else:
                # Every check passed. Now it is safe to change the data.
                total_fare = route["ticket_price"] * seats   # fare = price x seats
                booking_id = generate_booking_id()

                booking = {
                    "booking_id": booking_id,
                    "passenger_name": name,
                    "passenger_age": age,
                    "route_id": route_id,
                    "seats": seats,
                    "total_fare": total_fare,
                    "status": "Confirmed"
                }
                bookings.append(booking)                                     # 1) create the record
                route["available_seats"] = route["available_seats"] - seats  # 2) subtract the seats once

                print()
                print("Booking confirmed!")
                print("Booking ID:", booking_id)
                print("Passenger:", name)
                print("Route:", route_id, "(" + describe_journey(route) + ")")
                print("Seats booked:", seats)
                print("Fare per seat: Rs.", route["ticket_price"])
                print("Total fare: Rs.", total_fare)
                print("Status: Confirmed")


# ---- Menu option 4: View My Bookings -------------------------------

def view_bookings():
    print()
    print("MY BOOKINGS")
    if len(bookings) == 0:
        print("No bookings found.")
        return

    print("No. | Booking | Passenger | Route | Seats | Fare | Status")
    # Cancelled bookings are shown too: history is kept.
    for number, booking in enumerate(bookings, start=1):
        route = find_route(booking["route_id"])
        print(number, "|", booking["booking_id"], "|", booking["passenger_name"], "|",
              booking["route_id"], "(" + describe_journey(route) + ") |",
              booking["seats"], "|", "Rs." + str(booking["total_fare"]), "|",
              booking["status"])


# ---- Menu option 5: Cancel a Booking -------------------------------

def cancel_booking():
    print()
    print("CANCEL A BOOKING")
    booking_id = input("Enter booking ID to cancel: ").strip().upper()

    # Search the bookings list for this ID.
    for booking in bookings:
        if booking["booking_id"] == booking_id:
            if booking["status"] == "Confirmed":
                # Restore the seats and change the status together, as one operation.
                route = find_route(booking["route_id"])
                route["available_seats"] = route["available_seats"] + booking["seats"]
                booking["status"] = "Cancelled"   # the record stays; only the status changes

                print("Booking", booking_id, "has been cancelled.")
                print("Seats restored:", booking["seats"])
                print("Route", route["route_id"], "available seats:", route["available_seats"])
            else:
                # Already Cancelled: do nothing, so seats can never be restored twice.
                print("Booking", booking_id, "is already cancelled. No seats were restored.")
            break  # the booking was found, so stop searching
    else:
        # for-else: runs only if the loop ended WITHOUT "break", i.e. the ID does not exist.
        print("Booking ID", booking_id, "was not found. Nothing was changed.")


# ---- Menu option 6: View Seat Availability -------------------------

def show_availability():
    print()
    print("SEAT AVAILABILITY")
    print("1. All routes")
    print("2. Only routes with seats available")
    choice = input("Enter your choice: ").strip()
    if choice == "1":
        only_available = False
    elif choice == "2":
        only_available = True
    else:
        print("Invalid choice. Please enter 1 or 2.")
        return

    print()
    print("Route | Journey | Available | Total | Status")
    shown_count = 0
    for route in routes:
        if only_available and route["available_seats"] == 0:
            continue  # skip fully booked routes when the filter is on

        # Work out the status from how many seats remain.
        if route["available_seats"] == route["total_seats"]:
            status = "Available"
        elif route["available_seats"] > 0:
            status = "Partially Booked"
        else:
            status = "Fully Booked"

        shown_count = shown_count + 1
        print(route["route_id"], "|", describe_journey(route), "|",
              route["available_seats"], "|", route["total_seats"], "|", status)

    if shown_count == 0:
        print("No routes with seats available.")


# ---- Menu option 7: Generate Booking Report ------------------------

def generate_report():
    # Total bookings created counts EVERY record, cancelled ones included.
    total_created = len(bookings)

    # Count the Confirmed and Cancelled records.
    confirmed_count = 0
    cancelled_count = 0
    for booking in bookings:
        if booking["status"] == "Confirmed":
            confirmed_count = confirmed_count + 1
        elif booking["status"] == "Cancelled":
            cancelled_count = cancelled_count + 1

    # Group the CONFIRMED bookings by route with nested loops.
    # Outer loop = each route. Inner loop = look at every booking for that route.
    reserved_seats = 0
    confirmed_revenue = 0
    fully_booked_routes = 0
    for route in routes:
        for booking in bookings:
            if booking["route_id"] == route["route_id"] and booking["status"] == "Confirmed":
                reserved_seats = reserved_seats + booking["seats"]
                confirmed_revenue = confirmed_revenue + booking["total_fare"]

        # A route with zero available seats is fully booked.
        if route["available_seats"] == 0:
            fully_booked_routes = fully_booked_routes + 1

    print()
    print("BOOKING REPORT")
    print("Total bookings created:", total_created)
    print("Confirmed bookings:", confirmed_count)
    print("Cancelled bookings:", cancelled_count)
    print("Seats currently reserved:", reserved_seats)
    print("Confirmed revenue: Rs.", confirmed_revenue)
    print("Fully booked routes:", fully_booked_routes)


# --------------------------------------------------------------------
# SECTION 5: MAIN PROGRAM LOOP
# --------------------------------------------------------------------

def main():
    # The while loop repeats until the user chooses Exit (option 8).
    while True:
        show_menu()
        choice_text = input("Enter your choice: ").strip()

        # Only digits can be turned into a number. Anything else (such as
        # "abc", "3.5", "-1" or an empty answer) is rejected safely.
        if not choice_text.isdecimal():
            print("Invalid input. Please enter a number from 1 to 8.")
            continue  # go back to the top of the loop and show the menu again

        choice = int(choice_text)

        if choice == 1:
            show_routes_menu()
        elif choice == 2:
            search_routes()
        elif choice == 3:
            book_ticket()
        elif choice == 4:
            view_bookings()
        elif choice == 5:
            cancel_booking()
        elif choice == 6:
            show_availability()
        elif choice == 7:
            generate_report()
        elif choice == 8:
            print("Thank you for using the Smart Bus Reservation System. Goodbye!")
            break  # leave the while loop, which ends the program
        else:
            print("Unsupported choice. Please enter a number from 1 to 8.")


# Start the program. (In Jupyter you can also run this file with: %run main.py)
main()



SMART BUS RESERVATION SYSTEM
1. View Available Routes
2. Search for a Route
3. Book a Ticket
4. View My Bookings
5. Cancel a Booking
6. View Seat Availability
7. Generate Booking Report
8. Exit
